# Notebook 56: Single Condition Entry Tests

**Question:** What if we just use STH-SOPR < 1 alone?

Test all single conditions to see which has the most predictive power.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path

DATA_DIR = Path("../data")
HOURLY_DIR = DATA_DIR / "hourly"

In [ ]:
# Load data
def load_hourly():
    price = pd.read_parquet(HOURLY_DIR / "price.parquet")
    sopr = pd.read_parquet(HOURLY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(HOURLY_DIR / "sopr_sth.parquet")
    realized_loss = pd.read_parquet(HOURLY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    return df

def add_zscore(df, window):
    df = df.copy()
    df["rl_mean"] = df["realized_loss"].rolling(window=window, min_periods=window//2).mean()
    df["rl_std"] = df["realized_loss"].rolling(window=window, min_periods=window//2).std()
    df["rl_zscore"] = (df["realized_loss"] - df["rl_mean"]) / df["rl_std"]
    return df

df_full = load_hourly()
df_full = add_zscore(df_full, 365 * 24)

START = "2019-01-01"
df = df_full[df_full.index >= START].dropna()
years = (df.index.max() - df.index.min()).days / 365.25

print(f"Data: {len(df):,} hourly bars ({years:.1f} years)")

In [ ]:
def get_metrics(pf, years):
    trades = pf.trades.records_readable
    if len(trades) == 0:
        return None
    
    total_return = pf.total_return() * 100
    durations = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dropna()
    avg_days = durations.mean().total_seconds() / 86400 if len(durations) > 0 else 0
    
    winning = trades[trades["PnL"] > 0]
    losing = trades[trades["PnL"] < 0]
    
    return {
        "return": total_return,
        "cagr": ((1 + total_return/100) ** (1/years) - 1) * 100,
        "sharpe": pf.sharpe_ratio(),
        "max_dd": pf.max_drawdown() * 100,
        "trades": len(trades),
        "trades_yr": len(trades) / years,
        "avg_days": avg_days,
        "win_rate": (trades["PnL"] > 0).mean() * 100,
        "profit_factor": abs(winning["PnL"].sum() / losing["PnL"].sum()) if len(losing) > 0 and losing["PnL"].sum() != 0 else np.inf,
    }

def run_backtest(df, entry_cond, trail=0.12):
    entry = entry_cond & ~entry_cond.shift(1).fillna(False)
    if entry.sum() == 0:
        return None
    return vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail,
        sl_trail=True,
        freq="1h",
        init_cash=10000,
        fees=0.001
    )

## 1. Single Condition Tests

In [ ]:
# Define all conditions
conditions = {
    # Single conditions
    "STH-SOPR < 1 only": df["sopr_sth"] < 1,
    "SOPR < 1 only": df["sopr"] < 1,
    "RL z-score > 0.5 only": df["rl_zscore"] > 0.5,
    
    # Tighter single conditions
    "STH-SOPR < 0.98": df["sopr_sth"] < 0.98,
    "STH-SOPR < 0.95": df["sopr_sth"] < 0.95,
    "SOPR < 0.98": df["sopr"] < 0.98,
    "SOPR < 0.95": df["sopr"] < 0.95,
    "RL z-score > 1.0": df["rl_zscore"] > 1.0,
    "RL z-score > 1.5": df["rl_zscore"] > 1.5,
    
    # Original for comparison
    "ALL 3 (Original)": (df["sopr"] < 1) & (df["sopr_sth"] < 1) & (df["rl_zscore"] > 0.5),
}

print("Condition frequency:")
for name, cond in conditions.items():
    print(f"  {name:<25}: {cond.mean()*100:5.1f}%")

In [ ]:
print("\n" + "="*130)
print("SINGLE CONDITION BACKTEST RESULTS (1H @ 12% Trail)")
print("="*130)
print(f"\n{'Condition':<25} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'Days':>8} {'Win%':>8}")
print("-"*130)

results = {}
for name, cond in conditions.items():
    pf = run_backtest(df, cond)
    if pf:
        m = get_metrics(pf, years)
        results[name] = {"m": m, "pf": pf}
        print(f"{name:<25} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_yr']:>8.1f} {m['avg_days']:>8.1f} {m['win_rate']:>7.0f}%")
    else:
        print(f"{name:<25} {'No trades':>10}")

## 2. Forward Test (Past Year)

In [ ]:
FORWARD_START = "2024-01-15"
FORWARD_END = "2025-01-15"

df_fwd = df[(df.index >= FORWARD_START) & (df.index <= FORWARD_END)]
fwd_years = (df_fwd.index.max() - df_fwd.index.min()).days / 365.25
bh_return = (df_fwd["price"].iloc[-1] / df_fwd["price"].iloc[0] - 1) * 100

# Key conditions to forward test
fwd_conditions = {
    "STH-SOPR < 1 only": df_fwd["sopr_sth"] < 1,
    "STH-SOPR < 0.98": df_fwd["sopr_sth"] < 0.98,
    "STH-SOPR < 0.95": df_fwd["sopr_sth"] < 0.95,
    "SOPR < 1 only": df_fwd["sopr"] < 1,
    "RL z-score > 0.5 only": df_fwd["rl_zscore"] > 0.5,
    "ALL 3 (Original)": (df_fwd["sopr"] < 1) & (df_fwd["sopr_sth"] < 1) & (df_fwd["rl_zscore"] > 0.5),
}

print("\n" + "="*130)
print(f"FORWARD TEST: {FORWARD_START} to {FORWARD_END} (B&H: {bh_return:+.1f}%)")
print("="*130)
print(f"\n{'Condition':<25} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'Days':>8} {'Win%':>8}")
print("-"*130)

fwd_results = {}
for name, cond in fwd_conditions.items():
    pf = run_backtest(df_fwd, cond)
    if pf:
        m = get_metrics(pf, fwd_years)
        fwd_results[name] = {"m": m, "pf": pf}
        beat_bh = "✅" if m['return'] > bh_return else "❌"
        print(f"{name:<25} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_yr']:>8.1f} {m['avg_days']:>8.1f} {m['win_rate']:>7.0f}% {beat_bh}")
    else:
        print(f"{name:<25} {'No trades':>10}")

## 3. STH-SOPR Deep Dive

In [ ]:
# Test different STH-SOPR thresholds
sth_thresholds = [1.02, 1.01, 1.00, 0.99, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92, 0.90]

print("\n" + "="*130)
print("STH-SOPR THRESHOLD OPTIMIZATION (Full History)")
print("="*130)
print(f"\n{'Threshold':<15} {'Frequency':>10} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'Trades':>8} {'Tr/Yr':>8} {'Win%':>8}")
print("-"*100)

sth_results = {}
for thresh in sth_thresholds:
    cond = df["sopr_sth"] < thresh
    freq = cond.mean() * 100
    pf = run_backtest(df, cond)
    if pf:
        m = get_metrics(pf, years)
        sth_results[thresh] = m
        print(f"STH < {thresh:<8} {freq:>9.1f}% {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['trades']:>8} {m['trades_yr']:>8.1f} {m['win_rate']:>7.0f}%")

In [ ]:
# Visualize threshold impact
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

thresholds = list(sth_results.keys())
returns = [sth_results[t]["return"] for t in thresholds]
sharpes = [sth_results[t]["sharpe"] for t in thresholds]
trades = [sth_results[t]["trades_yr"] for t in thresholds]
win_rates = [sth_results[t]["win_rate"] for t in thresholds]

axes[0,0].plot(thresholds, returns, 'b-o')
axes[0,0].set_xlabel("STH-SOPR Threshold")
axes[0,0].set_ylabel("Total Return (%)")
axes[0,0].set_title("Return vs Threshold")
axes[0,0].invert_xaxis()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(thresholds, sharpes, 'g-o')
axes[0,1].set_xlabel("STH-SOPR Threshold")
axes[0,1].set_ylabel("Sharpe Ratio")
axes[0,1].set_title("Sharpe vs Threshold")
axes[0,1].invert_xaxis()
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(thresholds, trades, 'r-o')
axes[1,0].set_xlabel("STH-SOPR Threshold")
axes[1,0].set_ylabel("Trades per Year")
axes[1,0].set_title("Trade Frequency vs Threshold")
axes[1,0].invert_xaxis()
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(thresholds, win_rates, 'm-o')
axes[1,1].set_xlabel("STH-SOPR Threshold")
axes[1,1].set_ylabel("Win Rate (%)")
axes[1,1].set_title("Win Rate vs Threshold")
axes[1,1].invert_xaxis()
axes[1,1].axhline(y=50, color='black', linestyle='--', alpha=0.5)
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Summary

In [ ]:
print("\n" + "="*80)
print("SUMMARY: SINGLE CONDITION ANALYSIS")
print("="*80)

# Best single condition
single_conds = ["STH-SOPR < 1 only", "SOPR < 1 only", "RL z-score > 0.5 only"]
best_single = max(single_conds, key=lambda x: results.get(x, {}).get("m", {}).get("return", -999))

orig = results["ALL 3 (Original)"]["m"]
single = results[best_single]["m"]

print(f"\n{'Metric':<20} {'ALL 3 (Original)':>18} {best_single:>25}")
print("-"*70)
print(f"{'Return':<20} {orig['return']:>+17.0f}% {single['return']:>+24.0f}%")
print(f"{'CAGR':<20} {orig['cagr']:>+17.1f}% {single['cagr']:>+24.1f}%")
print(f"{'Sharpe':<20} {orig['sharpe']:>18.2f} {single['sharpe']:>25.2f}")
print(f"{'Trades/Year':<20} {orig['trades_yr']:>18.1f} {single['trades_yr']:>25.1f}")
print(f"{'Win Rate':<20} {orig['win_rate']:>17.0f}% {single['win_rate']:>24.0f}%")

print(f"\n{'='*70}")
print("VERDICT:")
if single['return'] > orig['return'] * 0.8 and single['trades_yr'] > orig['trades_yr'] * 1.5:
    print(f"  ✅ {best_single} is a viable simpler alternative")
    print(f"     Trade-off: More trades, possibly lower quality")
else:
    print(f"  ❌ ALL 3 conditions still optimal")
    print(f"     Single conditions lose too much edge")